In [1]:
import torch
import numpy as np
import random
import torchaudio
import os
import glob
from pathlib import Path
import librosa
from torch.utils.data import Dataset, random_split, DataLoader
import gc


# --- SET YOUR KAGGLE PATHS ---
INPUT_BASE = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
WORKING_BASE = '/kaggle/working'

from IPython.display import Audio, clear_output
import pandas as pd


In [3]:
test_df = pd.read_csv(INPUT_BASE+'/test.csv')

In [4]:
folder_index = {i:os.listdir('/'.join([INPUT_BASE, 'genres_stems', i])) for i in os.listdir(INPUT_BASE+'/genres_stems')}


STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
OUTPUT_PATH = os.path.join(WORKING_BASE, 'synthetic_mashups/train')

# Default random setting for
def seed_everything(seed=42):
    """Locks all random seeds for absolute reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True 
        torch.backends.cudnn.benchmark = False

seed_everything(42)




In [5]:
genres = os.listdir(INPUT_BASE+'/genres_stems')

In [6]:
from tqdm import tqdm, trange

In [7]:
tf = INPUT_BASE + '/genres_stems/disco/'+folder_index['disco'][5]
os.listdir(tf)

['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']

In [8]:
# Not Used due to slow computation and excessive ram usage

def find_tempo(waveform, sr):
    audio = waveform.squeeze().numpy().astype(np.float32)   # (T,)
    tempo, _ = librosa.beat.beat_track(y=audio, sr=sr)
    return max(float(np.atleast_1d(tempo)[0]), 1.0)

def match_tempo(stems, sr):
    target_length = stems[0].shape[1]
    tempos = [find_tempo(s, sr) for s in stems]
    target_tempo = float(np.median(tempos))                 

    synced = []
    for i, (stem, src_tempo) in enumerate(zip(stems, tempos)):
        if abs(src_tempo - target_tempo) < 0.5:             
            synced.append(stem)
            continue

        rate = src_tempo / target_tempo
        audio = stem.squeeze().numpy().astype(np.float32)   

        stretched = librosa.effects.time_stretch(audio, rate=rate)

        if len(stretched) > target_length:
            stretched = stretched[:target_length]
        elif len(stretched) < target_length:
            stretched = np.pad(stretched, (0, target_length - len(stretched)))

        result = torch.from_numpy(stretched).unsqueeze(0)   # (1, T)
        del audio, stretched                                 # free immediately
        synced.append(result)

    return synced
        

In [9]:
target_sr = 16000
target_duration = 30
target_samples = target_duration * target_sr            



def load_aud(path, target_samples=target_samples, sr=target_sr):
    wav, orig_sr = torchaudio.load(path)
    if orig_sr != sr:
        wav = torchaudio.functional.resample(wav, orig_sr, sr)

    wav = wav.mean(dim=0)
    # Cut for longer audio
    if wav.shape[0] > target_samples:
        start = torch.randint(0, wav.shape[0] - target_samples, (1,)).item()
        wav = wav[start:start + target_samples]
    # pad for shorter audio
    elif wav.shape[0] < target_samples:
        wav = torch.nn.functional.pad(wav, (0, target_samples - wav.shape[0]))

    return wav 

In [10]:
# Generation of randomly mixed mashup with noise

def generate_synthetic_dataset(stems_dir, noise_dir, output_dir, samples_per_genre=50, target_sr=16000, duration=30):
    genres = os.listdir(INPUT_BASE+'/genres_stems')
    
    target_length = target_sr * duration
    
    # Get noise files from input
    noise_files = glob.glob(os.path.join(noise_dir, '**', '*.wav'), recursive=True)
    
    for genre in genres:
        clear_output()
        print(f'Genre: {genre}: {genres.index(genre)+1}/10')

        genre_out_dir = Path(output_dir) / genre
        genre_out_dir.mkdir(parents=True, exist_ok=True)
        
        song_folders = glob.glob(os.path.join(stems_dir, genre, '*'))
        if not song_folders: 
            print(f"Warning: No songs found for genre {genre}")
            continue
        
        for i in trange(samples_per_genre):
            chosen_songs = random.sample(song_folders, 4)
            stems = []
            stem_types = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
            
            for song, stem_type in zip(chosen_songs, stem_types):
                stem_path = os.path.join(song, stem_type)
                if os.path.exists(stem_path):
                    waveform, sr = torchaudio.load(stem_path)
                    
                    # Resampling added to load_aud function
                    # if sr != target_sr:
                    #     resampler = torchaudio.transforms.Resample(sr, target_sr)
                    #     waveform = resampler(waveform)

                    if waveform.shape[1] > target_length:
                        waveform = waveform[:, :target_length]
                    elif waveform.shape[1] < target_length:
                        waveform = torch.nn.functional.pad(waveform, (0, target_length - waveform.shape[1]))
                    stems.append(waveform)
            
            # stems = match_tempo(stems, target_sr)

            # Mixing audios
            if len(stems) == 4:
                mashup = torch.stack(stems).sum(dim=0)
                mashup = mashup.mean(dim=0, keepdim=True)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                # Adding Noise at random position with volume between 10 and 40 %
                noise_file = random.choice(noise_files)
                noise, _ = torchaudio.load(noise_file)
                noise = noise.mean(dim=0, keepdim=True)
                
                if noise.shape[1] > target_length:
                    noise = noise[:, :target_length]
                    
                start_idx = random.randint(0, target_length - noise.shape[1])
                intensity = random.uniform(0.1, 0.4)
                
                # mixing noise and normalizing volume
                mashup[:, start_idx:start_idx + noise.shape[1]] += (noise * intensity)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                # Save to /kaggle/working/
                out_path = genre_out_dir / f"mashup_{i:03d}.wav"
                torchaudio.save(str(out_path), mashup, target_sr)
    
    gc.collect()

In [44]:
# skip generation if custom uploaded dataset available 
if os.path.isdir('/kaggle/input/datasets/pirrabur/synthetic/kaggle/working/synthetic_mashups/train'):
    OUTPUT_PATH = '/kaggle/input/datasets/pirrabur/synthetic/kaggle/working/synthetic_mashups/train'
else:
    generate_synthetic_dataset(STEMS_PATH, NOISE_PATH, OUTPUT_PATH, samples_per_genre=100)

In [34]:
Audio(OUTPUT_PATH + '/jazz/mashup_009.wav')

In [46]:
idx_g = dict(enumerate(set(genres)))
g_idx = {j:i for i,j in idx_g.items()}

In [35]:
import wandb

In [36]:
wandb.login(key="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")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [66]:
import torch
from torch import nn
from transformers import ASTModel, ASTConfig, ASTFeatureExtractor
base_model = "MIT/ast-finetuned-audioset-10-10-0.4593"

class ASTClassifier(nn.Module):
    def __init__(self, num_classes=10, backbone=base_model):
        super().__init__()
        self.backbone = ASTModel.from_pretrained(backbone)
        
        hidden = self.backbone.config.hidden_size 

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, 1024),
            nn.Tanh(),
            nn.LayerNorm(1024),
            nn.Linear(1024, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, input_values):
        outputs = self.backbone(input_values)
        # not pooled, using only first token out instead of avg/max/sum
        pooled_output = outputs.last_hidden_state[:, 0, :] 
        
        return self.classifier(pooled_output)


class SyntheticDataset(Dataset):
    def __init__(self, output_dir):
        self.samples = []
        self.feature_extractor = ASTFeatureExtractor.from_pretrained(base_model)
        for genre in genres:
            genre_dir = Path(output_dir) / genre
            for wav_path in sorted(genre_dir.glob("*.wav")):
                self.samples.append((str(wav_path), g_idx[genre]))
        
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        wav = load_aud(path)
        inputs = self.feature_extractor(wav, 
                                        sampling_rate=target_sr, 
                                        return_tensors='pt',
                                        max_length=1024)['input_values'].squeeze(0)
        return inputs, label

class TestMashupDataset(Dataset):
    def __init__(self, mashups_dir, test_csv):
        self.df = pd.read_csv(test_csv)
        self.mashups_dir = Path(mashups_dir)
        self.feature_extractor = ASTFeatureExtractor.from_pretrained(base_model)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        fname = row.get("filename", f"{row['id']}.wav")
        wav   = load_aud(str(self.mashups_dir / fname))
        inputs = self.feature_extractor(wav, 
                                        sampling_rate=target_sr, 
                                        return_tensors='pt',
                                        max_length=1024)['input_values'].squeeze(0)
        return inputs, str(row["id"])

In [67]:
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from transformers import get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report
from tqdm import tqdm

SAVE_PATH     = "/kaggle/working/best_ast.pt"
EPOCHS        = 20
LR            = 5e-4
WEIGHT_DECAY  = 1e-4
VAL_SPLIT     = 0.1
# limiting numeber of steps per backpropogation as small dataset, and lesser compute
ACCUM_STEPS   = 4
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"


def collate_fn(batch):
    inputs, labels = zip(*batch)

    inputs = torch.stack(inputs)
    labels = torch.tensor(labels, dtype=torch.long)
    return inputs, labels

In [68]:
BATCH_SIZE=16
wandb.init(
    project="messy-mashup",
    name="ast-finetuned-3",
    config={
        "backbone":     base_model,
        "batch_size":   BATCH_SIZE,
        "epochs":       EPOCHS,
        "lr":           LR,
        "accum_steps":  ACCUM_STEPS,
        "label_smoothing": 0.1,
    }
)

full_ds = SyntheticDataset(OUTPUT_PATH)
print(f"Total samples: {len(full_ds)}")

val_n   = max(1, int(len(full_ds) * VAL_SPLIT))
train_n = len(full_ds) - val_n
train_ds, val_ds = random_split(
    full_ds, [train_n, val_n],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True,
    collate_fn=collate_fn
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True,
    collate_fn=collate_fn
)



Total samples: 1000


In [69]:
model = ASTClassifier(num_classes=10).to(DEVICE)
model = nn.DataParallel(model)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

for name, param in model.named_parameters():
    if 'intermediate' in name or 'classifier' in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)

total_steps  = (len(train_loader) // ACCUM_STEPS) * EPOCHS
warmup_steps = total_steps // 10

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
scaler = GradScaler('cuda')

wandb.watch(model, log="gradients", log_freq=50)

best_f1     = 0.0
global_step = 0

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ASTModel LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                         | Status     |  | 
----------------------------+------------+--+-
classifier.layernorm.weight | UNEXPECTED |  | 
classifier.dense.bias       | UNEXPECTED |  | 
classifier.dense.weight     | UNEXPECTED |  | 
classifier.layernorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [70]:
for epoch in range(1, EPOCHS + 1):

    # ── train ───────────────────────────────────────────────────────
    model.train()
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for step, (inputs, labels) in enumerate(
            tqdm(train_loader, desc=f"Ep {epoch:02d} train"), start=1):

        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE)

        with autocast('cuda'):
            logits = model(inputs)
            loss   = criterion(logits, labels) / ACCUM_STEPS

        scaler.scale(loss).backward()
        running_loss += loss.item() * ACCUM_STEPS

        if step % ACCUM_STEPS == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

            wandb.log({
                "train/batch_loss": loss.item() * ACCUM_STEPS,
                "train/lr":         scheduler.get_last_lr()[0],
            }, step=global_step)

    # ── validate ────────────────────────────────────────────────────
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Ep {epoch:02d} val  "):
            inputs = inputs.to(DEVICE)
            with autocast('cuda'):
                preds = model(inputs).argmax(dim=-1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels.tolist())

    val_f1       = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    per_class_f1 = f1_score(all_labels, all_preds, average=None,    zero_division=0)
    avg_loss     = running_loss / len(train_loader)

    epoch_metrics = {
        "epoch":              epoch,
        "train/epoch_loss":   avg_loss,
        "val/macro_f1":       val_f1,
    }
    for genre, f1 in zip(genres, per_class_f1):
        epoch_metrics[f"val/f1_{genre}"] = f1

    wandb.log(epoch_metrics, step=global_step)
    print(f"  loss: {avg_loss:.4f}  |  val macro-F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"  saved (best F1={best_f1:.4f})")

        artifact = wandb.Artifact("ast-best-model", type="model",
                                  description=f"val macro-F1: {best_f1:.4f}")
        artifact.add_file(SAVE_PATH)
        wandb.log_artifact(artifact)

    if epoch % 5 == 0:
        print(classification_report(
            all_labels, all_preds,
            target_names=genres, zero_division=0
        ))

    torch.cuda.empty_cache()

wandb.log({
    "val/confusion_matrix": wandb.plot.confusion_matrix(
        probs=None, y_true=all_labels,
        preds=all_preds, class_names=genres
    )
})

wandb.finish()
print(f"\nDone. Best val macro-F1: {best_f1:.4f}")

Ep 01 val  : 100%|██████████| 7/7 [00:05<00:00,  1.17it/s]


  loss: 1.9073  |  val macro-F1: 0.7124
  saved (best F1=0.7124)


Ep 02 val  : 100%|██████████| 7/7 [00:03<00:00,  1.89it/s]


  loss: 0.9180  |  val macro-F1: 0.7904
  saved (best F1=0.7904)


Ep 03 val  : 100%|██████████| 7/7 [00:03<00:00,  1.83it/s]


...

  loss: 0.5208  |  val macro-F1: 0.9293


Ep 20 val  : 100%|██████████| 7/7 [00:03<00:00,  1.84it/s]


  loss: 0.5209  |  val macro-F1: 0.9293
              precision    recall  f1-score   support

       disco       0.67      1.00      0.80         6
       metal       1.00      1.00      1.00        10
      reggae       0.83      0.91      0.87        11
       blues       1.00      1.00      1.00        11
        rock       1.00      0.92      0.96        13
   classical       1.00      1.00      1.00        16
        jazz       0.80      1.00      0.89         4
      hiphop       1.00      0.80      0.89        10
     country       1.00      0.88      0.93         8
         pop       1.00      0.91      0.95        11

    accuracy                           0.94       100
   macro avg       0.93      0.94      0.93       100
weighted avg       0.95      0.94      0.94       100



epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train/batch_loss,█▇▅▃▁▁▁▂▁▁▁▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/epoch_loss,█▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/lr,▂▃▆▆▆███████▇▇▇▇▆▅▅▅▄▄▄▄▃▃▃▃▂▂▁▁▁▁▁▁▁▁▁▁
val/f1_blues,▁▇▂▇▆█▇█▇▆█▆████████
val/f1_classical,█▁██▁▁██████████████
val/f1_country,▁▆▆▆▆▄▄▆▄▄▆█▆▆▆▆▆▆▆▆
val/f1_disco,▂▁▁█▇███▇▇▃▇▅▅▅▄▄▄▄▄
val/f1_hiphop,▃▁▃█▇▃▅▇▅▅▇▅▇▇▇▇▇▇▇▇
val/f1_jazz,▄▄▄▁▄▆▃▅▆▆█▆▆▆▆▆▆▆▆▆
+5,...



Done. Best val macro-F1: 0.9435


In [90]:
# !zip -r output.zip /kaggle/working/synthetic_mashups

  adding: kaggle/working/synthetic_mashups/ (stored 0%)
  adding: kaggle/working/synthetic_mashups/train/ (stored 0%)
  adding: kaggle/working/synthetic_mashups/train/classical/ (stored 0%)
  adding: kaggle/working/synthetic_mashups/train/classical/mashup_092.wav (deflated 9%)
  adding: kaggle/working/synthetic_mashups/train/classical/mashup_073.wav (deflated 7%)
  adding: kaggle/working/synthetic_mashups/train/classical/mashup_076.wav (deflated 9%)
  adding: kaggle/working/synthetic_mashups/train/classical/mashup_035.wav (deflated 8%)
  adding: kaggle/working/synthetic_mashups/train/classical/mashup_087.wav (deflated 6%)
  adding: kaggle/working/synthetic_mashups/train/classical/mashup_039.wav (deflated 6%)
  adding: kaggle/working/synthetic_mashups/train/classical/mashup_050.wav (deflated 9%)
  adding: kaggle/working/synthetic_mashups/train/classical/mashup_074.wav (deflated 9%)
  adding: kaggle/working/synthetic_mashups/train/classical/mashup_053.wav (deflated 11%)
  adding: kaggle/

model(fe(load_aud('/kaggle/working/synthetic_mashups/train/jazz/mashup_009.wav'), sampling_rate=16000, return_tensors='pt')['input_values'].to('cuda'))

In [71]:
test_ds = TestMashupDataset(INPUT_BASE,INPUT_BASE + '/test.csv')

In [73]:
test_loader = DataLoader(test_ds, batch_size=32)

In [78]:
model.to('cuda:1')

ASTClassifier(
  (backbone): ASTModel(
    (embeddings): ASTEmbeddings(
      (patch_embeddings): ASTPatchEmbeddings(
        (projection): Conv2d(1, 768, kernel_size=(16, 16), stride=(10, 10))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ASTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ASTLayer(
          (attention): ASTAttention(
            (attention): ASTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ASTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ASTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (intermediate_ac

In [74]:
all_ids = []
all_outputs = []
for inputs, ids in tqdm(test_loader):
    out = [idx_g[i] for i in model(inputs.to('cuda:1')).argmax(axis=1).cpu().numpy()]
    all_outputs += out
    all_ids += list(ids)

100%|██████████| 95/95 [05:22<00:00,  3.40s/it]


In [75]:
len(all_outputs)

3020

In [77]:
pd.DataFrame({'id':all_ids, 'genre':all_outputs}).to_csv('submission.csv',index=False)

wandb: 
wandb: Find logs at: wandb/run-20260331_163225-g44btgi2/logs
